# Week 2 — Contacts vs Donors Analysis Plan

## Objective
Compare the Contacts list to the Great Futures donation records to:

1) Identify repeated donors
2) Determine how many contacts are also donors
3) Assess whether donors have at least one form of contact information

Registration files will NOT be merged at this stage per mentor guidance.

---

## Scope of Work

### 1. Build Contacts Working Table
- Clean and standardize identity/contact fields
- Check for duplicate contacts
- Prepare for donor matching

### 2. Build Great Futures Donations Table
- Combine:
  - Great Futures Donation Log
  - Great Futures 2024
  - Great Futures 2025
- Standardize column names
- Standardize donation date and amount fields
- Add source metadata

---

## Donor Matching Strategy

### Primary Donor Key:
Email (when available)

### Fallback Matching:
1) Phone
2) Name + Zip (if Email missing)

This multi-field approach is required due to incomplete identity fields discovered in Week 1.

---

## Repeated Donor Definition
A repeated donor is defined as:
The same donor key appearing in two or more donation records.

---

## Donor Contact Coverage Check
For each donor:
Determine whether at least one contact method exists:
- Email OR
- Phone OR
- Address

Goal:
Identify donors without any contact information.

---

## Deliverables (End of Week 2)

- Cleaned Contacts table
- Combined Donations table
- Repeated donor summary
- Contacts vs Donors comparison summary
- Donors missing contact information report

## A) Donor Contact Coverage

**Goal:**
For each donor, determine whether they have at least one usable contact method:
- email OR phone OR mailing address

**Output:**
A summary of donors missing all contact information.

## B) Repeated Donors

**Goal:**
Identify donors who appear multiple times in the donation records.

**Definition:**
A repeated donor = same donor key appears in 2+ donation records.

**Output:**
A count of repeated donors and a small table of top repeat donors (by # donations).

## Step 0 — Create donations_all from cleaned donation files

In [1]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("..").resolve()
RAW_DIR = BASE_DIR / "data" / "raw"
CLEAN_DIR = BASE_DIR / "data" / "cleaned"

print("BASE_DIR:", BASE_DIR)
print("CLEAN_DIR exists:", CLEAN_DIR.exists())

BASE_DIR: /home/ce5b4deb-dc9a-4dd7-9a99-cd66f2b354ff/BGC_donor_project
CLEAN_DIR exists: True


In [2]:
#0A: find donation files
donation_files = sorted(CLEAN_DIR.glob("cleaned_donation_*.csv"))
len(donation_files), [p.name for p in donation_files]

(4,
 ['cleaned_donation_source03_donation_great_futures_2024.csv',
  'cleaned_donation_source04_donation_great_futures_2024.csv',
  'cleaned_donation_source04_donation_great_futures_2025.csv',
  'cleaned_donation_source08_fundraising_auction_sold_items.csv'])

In [3]:
# filter out the source04 file
exclude_name = "cleaned_donation_source04_donation_great_futures_2024.csv"
donation_files_dedup = [p for p in donation_files if p.name != exclude_name]

len(donation_files), len(donation_files_dedup), [p.name for p in donation_files_dedup]

(4,
 3,
 ['cleaned_donation_source03_donation_great_futures_2024.csv',
  'cleaned_donation_source04_donation_great_futures_2025.csv',
  'cleaned_donation_source08_fundraising_auction_sold_items.csv'])

In [4]:
# cell 0B: load + combine
donation_dfs = []
for p in donation_files_dedup:
    tmp = pd.read_csv(p)
    tmp["cleaned_file"] = p.name
    donation_dfs.append(tmp)

donations_all = pd.concat(donation_dfs, ignore_index=True)
donations_all.shape

(276, 20)

## Step 1 — Create a Donor Key (Email → Phone fallback)

**Purpose:**

Create a consistent identifier to count repeated donors without requiring a donor_id.

**Rule:**

- Use Email when available

- Else use Phone

This supports repeated-donor counting and contact coverage reporting.

In [5]:
donations_all.columns

Index(['akel', 'louise', '250', 'unnamed_3', 'unnamed_4', 'unnamed_5',
       'source_system', 'source_category', 'cleaned_file', 'last_name',
       'first_name', 'amount', 'recurring', 'anonymous', 'email',
       'method_of_receipt', 'date', 'items_purchased__2023', 'unnamed_1',
       'unnamed_2'],
      dtype='object')

In [6]:
donations_all["email_clean"] = donations_all["email"].astype("string").str.strip().str.lower()

donations_all["donor_key"] = donations_all["email_clean"]

In [7]:
donations_all["donor_key"].isna().mean()

0.9420289855072463

In [8]:
# A) Donor Contact Coverage (donation sources only)
donations_all["has_contact"] = donations_all["email_clean"].notna()
donations_all["has_contact"].value_counts(normalize=True)

has_contact
False    0.942029
True     0.057971
Name: proportion, dtype: float64

In [9]:
# B) Repeated donors (email-based)
donor_counts = donations_all["donor_key"].value_counts(dropna=True)
repeated_donors = (donor_counts > 1).sum()
repeated_donors

0

## Step 3 — Load Contacts cleaned file

In [10]:
contact_files = sorted(CLEAN_DIR.glob("cleaned_*contact*.csv"))
len(contact_files), [p.name for p in contact_files]

(1, ['cleaned_registration_source07_registration_contacts.csv'])

## Step 4 — Load contacts and inspect available identity fields

In [11]:
contacts = pd.read_csv(contact_files[0])
contacts.shape

(830, 15)

In [12]:
contacts.columns

Index(['first_name', 'last_name', 'business_name', 'business_address',
       'business_city', 'business_state', 'business_zip_code', 'address',
       'city', 'state', 'zip_code', 'phone_number', 'email', 'source_system',
       'source_category'],
      dtype='object')

## Step 5 — Matching Keys (Contacts vs Donations)

**Purpose:**
Create standardized match keys so donors can be linked to contacts using any contact method.

**Rules:**

- Normalize emails (lowercase + strip)

- Normalize phone (strip; later can format digits)

- Match priority: email first, phone second

In [13]:
# Donations: email key
donations_all["email_clean"] = donations_all["email"].astype("string").str.strip().str.lower()
donations_all["match_key"] = donations_all["email_clean"]

# Contacts: email + phone keys
contacts["email_clean"] = contacts["email"].astype("string").str.strip().str.lower()

phone_col = [c for c in contacts.columns if "phone" in c.lower()][0]
contacts["phone_clean"] = contacts[phone_col].astype("string").str.strip()

contacts["match_key"] = contacts["email_clean"].fillna(contacts["phone_clean"])

## Step 6 — Coverage check: do donations match to contacts?

In [14]:
contact_keys = set(contacts["match_key"].dropna())
donations_all["matched_to_contact"] = donations_all["match_key"].isin(contact_keys)

donations_all["matched_to_contact"].value_counts(normalize=True)

matched_to_contact
False    0.985507
True     0.014493
Name: proportion, dtype: float64

## Step 7 — Diagnose WHY matches are failing (fast + safe)

In [15]:
# 7A) Missing email rate in donations
donations_all["email_clean"].isna().mean()

0.9420289855072463

In [16]:
# 7B) Missing email rate in contacts
contacts["email_clean"].isna().mean()

0.7771084337349398

In [17]:
# 7C) How many contacts only have phone (no email)?
contacts["email_clean"].isna().mean(), contacts["phone_clean"].isna().mean()

(0.7771084337349398, 0.7807228915662651)

**Interpretation:**

Match rate is currently low because donation sources are missing email for most records and do not include phone.

**Plan**

Next step is to enrich donation records by linking them to contacts using additional identifiers
(e.g., name + zip/address) or by ensuring donation sources include a contact identifier.

## Conclusion

Our current donations sources don’t carry enough contact info to match donors to contacts reliably. We’ll need to enrich donations using the contacts list (phone/address/name+zip) or treat donations as transaction-only and link later.”

## Step 8 — Contacts Who Are Donors (Email Match)

**Purpose:**

Estimate overlap between contacts and donors based on shared email.

**Note:**

This is a lower-bound estimate because many donation records are missing email.

In [18]:
donor_emails = set(donations_all["email_clean"].dropna())
contacts["is_donor_by_email"] = contacts["email_clean"].isin(donor_emails)

contacts["is_donor_by_email"].value_counts(normalize=True)

is_donor_by_email
False    0.995181
True     0.004819
Name: proportion, dtype: float64

### Interpretation:

Even though donor_key is missing a lot, I can still identify repeat donors among the donors who DO have email. 

## Step 9 — Repeated donors among known emails

In [19]:
donor_counts = donations_all["email_clean"].value_counts(dropna=True)
repeated_donors = (donor_counts > 1).sum()
repeated_donors

0

### Interpretation: 

There are no repeated donors. Next, I will do a name - based match. 

**Plan:** 

1. Create **name_clean** in donations and contacts

2. If contacts has zip or address, I use Name + Zip as the stronger fallback match

3. If no zip, I do Name-only but I flag it as low confidence


## Step 10 — Identify name columns (donations + contacts)

In [20]:
donation_name_candidates = [c for c in donations_all.columns if any(k in c.lower() for k in ["name", "first", "last"])]
donation_name_candidates

['unnamed_3',
 'unnamed_4',
 'unnamed_5',
 'last_name',
 'first_name',
 'unnamed_1',
 'unnamed_2']

In [21]:
contact_name_candidates = [c for c in contacts.columns if any(k in c.lower() for k in ["name", "first", "last"])]
contact_name_candidates

['first_name', 'last_name', 'business_name']

## Step 11 — Build name_clean in donations

In [22]:
donations_all["first_clean"] = donations_all["first_name"].astype("string").str.strip().str.lower()
donations_all["last_clean"] = donations_all["last_name"].astype("string").str.strip().str.lower()

donations_all["name_clean"] = donations_all["first_clean"] + " " + donations_all["last_clean"]

In [23]:
donations_all["name_clean"].head()

0    <NA>
1    <NA>
2    <NA>
3    <NA>
4    <NA>
Name: name_clean, dtype: string

### Interpretation: 

In the first few rows, **first_name** and/or **last_name** are missing - so **name_clean** becomes NA.

In donation logs (names might be in a different column, or only present for some sources).

**Plan:**

Diagnose it cleanly.

## Step 11A — Check missingness for first_name and last_name

In [24]:
donations_all["first_name"].isna().mean(), donations_all["last_name"].isna().mean()

(0.7753623188405797, 0.7753623188405797)

**Interpretation:** 

This is high, therefore names are not populated for many records. 

**Plan:**

Scan for columns that contain "name" OR look like an unnamed column with data. 

## Step 11B — See if names are hiding in another column (including those unnamed ones)

In [25]:
name_like_cols = [c for c in donations_all.columns if "name" in c.lower()]
unnamed_cols = [c for c in donations_all.columns if str(c).startswith("unnamed")]

name_like_cols, unnamed_cols

(['unnamed_3',
  'unnamed_4',
  'unnamed_5',
  'last_name',
  'first_name',
  'unnamed_1',
  'unnamed_2',
  'name_clean'],
 ['unnamed_3', 'unnamed_4', 'unnamed_5', 'unnamed_1', 'unnamed_2'])

In [26]:
[(c, donations_all[c].isna().mean()) for c in unnamed_cols]

[('unnamed_3', 0.6086956521739131),
 ('unnamed_4', 0.9927536231884058),
 ('unnamed_5', 0.9963768115942029),
 ('unnamed_1', 0.6159420289855072),
 ('unnamed_2', 0.5507246376811594)]

**Interpretation:** 

unnamed_1, unnamed_2, and unnnamed_3 columns have missingness well below 1.0(they contain data). 

They likely came from:

* Merged headers in Excel
* Misaligned columns during export
* Different schema across donation files

**Plan:**

Expect those columns. 

## Step 12 — Inspect Unnamed Columns

**Purpose:**

Determine whether unnamed columns contain usable identity information
(e.g., names, addresses) or are artifacts from Excel formatting.

**Approach:**

- Preview sample values

- Measure missingness

- Decide whether to exclude or retain for matching

In [27]:
donations_all[["unnamed_1", "unnamed_2", "unnamed_3"]].head(10)

,unnamed_1,unnamed_2,unnamed_3
0,NaN,NaN,NaN
1,NaN,NaN,NaN
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,NaN,NaN,NaN
5,NaN,NaN,NaN
6,NaN,NaN,NaN
7,NaN,NaN,NaN
8,NaN,NaN,NaN
9,NaN,NaN,NaN


**Finding:**

Unnamed columns (unnamed_1, unnamed_2, unnamed_3) appear to contain mostly missing values
in the combined donation dataset.

**Conclusion:**

These columns are not reliable identity fields for donor matching at this stage.
They will be excluded from matching logic unless further inspection shows
structured information in a specific source.

# Step 13 — Name Field Coverage (Donation Sources)

**Purpose:**

Measure how often first_name and last_name are present in donation records.

**Goal:**

Determine whether name-based matching (Name + Zip fallback) is feasible.

In [28]:
has_name = donations_all["first_name"].notna() & donations_all["last_name"].notna()
has_name.mean()

0.21739130434782608

**Finding:**

Less than 20% of donation records contain both first_name and last_name.

**Impact:**

Name-based matching alone will have limited coverage.

**Conclusion:**

Donation records currently lack sufficient identity fields for reliable
automated donor matching.

**Matching must be enhanced during consolidation using:**

- Contact table enrichment

- Creation of a donor_id

## Week 2 Matching Readiness Summary

**Key constraint:**

Donation sources lack reliable identity fields:

- Email missing rate is >70%

- Phone column not present in donation sources

- First/last name available for <20% of donation rows

**Impact:**

- Automated matching of donors to contacts will have low coverage using donations alone.

- Current matches (email-based) represent a lower-bound estimate.

**Next step for consolidation:**

During consolidation, ensure donation records include at least one contact identifier:
email OR phone OR donor_id OR name + zip/address.

## Single Source of Truth — Required Fields Checklist (Donor + Donations)

## Donor Identity (at least ONE must be present)

- Email

- Phone

- Full Name + Zip (or Address)

- Donor ID (if created during consolidation)

## Donation Transaction Fields

- Donation date

- Donation amount

- Campaign/event name (if applicable)

- Source file (metadata)

# Step 14 — Check Zip/Address Availability for Fallback Matching

**Purpose:**

Enable Name + Zip (or Name + Address) matching when email is missing.

**Rule:**

Only use fallback matching when enough fields exist to reduce false matches.

In [31]:
don_zip_candidates = [c for c in donations_all.columns if "zip" in c.lower() or "postal" in c.lower()]
don_addr_candidates = [c for c in donations_all.columns if "address" in c.lower()]

con_zip_candidates = [c for c in contacts.columns if "zip" in c.lower() or "postal" in c.lower()]
con_addr_candidates = [c for c in contacts.columns if "address" in c.lower()]

don_zip_candidates, don_addr_candidates, con_zip_candidates, con_addr_candidates

([], [], ['business_zip_code', 'zip_code'], ['business_address', 'address'])

**Result (Zip/Address availability):**
- Zip and address fields exist in both Contacts and Donation sources.

**Interpretation:**
This enables a higher-confidence fallback match when email is missing by using:
- Name + Zip (preferred)
- Name + Address (optional/secondary)

**Next step:**
Create a standardized Name+Zip match key in the Contacts table and then attempt
fallback matching for donation records missing email.

# Step 15 — Create Contacts Name+Zip Match Key

**Purpose:**

Create a fallback match key for contacts when email is missing.

**Method:**

- Standardize first and last name

- Standardize zip (keep as string, trim whitespace)

- Combine into: name_zip_key = "first last|zip"

In [30]:
contact_name_cols = [c for c in contacts.columns if any(k in c.lower() for k in ["first", "last", "name"])]
contact_zip_col = [c for c in contacts.columns if "zip" in c.lower() or "postal" in c.lower()][0]

contact_name_cols, contact_zip_col

(['first_name', 'last_name', 'business_name'], 'business_zip_code')

**Result (Contacts Name+Zip key):**
- Contacts contain `first_name` and `last_name`, so I can build a standardized full name.
- A zip/postal column exists, so I can attach location information to reduce false matches.

**Match key definition:**
`name_zip_key = (first_name + last_name standardized) + "|" + (zip standardized)`

**Why this matters:**
Name-only matching is high-risk (many people share names).
Adding zip increases match confidence and helps link donors to contacts even when email is missing.

# Step 16 — Name-Based Fallback Matching (Review Only)

**Purpose:**

Donation sources do not include a zip/postal field, so Name+Zip matching is not possible.

**Approach:**

- Match by email where possible (high confidence)

- For records missing email, attempt name-only matching as a REVIEW LIST

- Do not automatically merge name-only matches

In [34]:
contacts["name_clean"] = (
    contacts["first_name"].astype("string").str.strip().str.lower()
    + " "
    + contacts["last_name"].astype("string").str.strip().str.lower()
)

In [35]:
donations_all["name_clean"] = (
    donations_all["first_name"].astype("string").str.strip().str.lower()
    + " "
    + donations_all["last_name"].astype("string").str.strip().str.lower()
)

In [43]:
contact_email_keys = set(contacts["email_clean"].dropna())
donations_all["matched_by_email"] = donations_all["email_clean"].isin(contact_email_keys)

In [41]:
contact_name_keys = set(contacts["name_clean"].dropna())

donations_all["possible_match_by_name_only"] = (
    ~donations_all["matched_by_email"]
    & donations_all["email_clean"].isna()
    & donations_all["name_clean"].isin(contact_name_keys)
)

In [38]:
donations_all[["matched_by_email", "possible_match_by_name_only"]].mean()

matched_by_email               0.014493
possible_match_by_name_only    0.101449
dtype: float64

**Results (Match Coverage):**
- `matched_by_email` ≈ 1.45%  
  High-confidence matches where donation email exists in the Contacts table.
- `possible_match_by_name_only` ≈ 10.14%  
  Low-confidence candidates where donation email is missing but donor name appears in Contacts.

**Interpretation:**
Email-based matching has very low coverage because donation records frequently lack email.
Name-only matching increases potential coverage but may contain false matches (shared names).

**Action for the team:**
- Use email matches as confirmed links.
- During consolidation, prioritize adding a stable donor identifier (donor_id) and/or ensuring donation records include a contact identifier (email/phone/address/zip).

# Step 17 — Build Name-Only Review List (Manual Verification)

**Purpose:**

Create a side-by-side table of donation records and matching contact records
based on name-only matching.

**Scope:**

- Include only donation rows flagged as possible_match_by_name_only

- Do NOT automatically merge these records

- Use this table for manual stakeholder review

**Goal:**

Enable safe verification before consolidation.

In [44]:
# Donation rows needing review
review_donations = donations_all[
    donations_all["possible_match_by_name_only"]
].copy()

In [45]:
# Merge with contacts on name_clean
review_table = review_donations.merge(
    contacts,
    on="name_clean",
    how="left",
    suffixes=("_donation", "_contact")
)

review_table.shape

(28, 49)

In [46]:
# useful columns selected
review_table_subset = review_table[
    [
        "first_name_donation",
        "last_name_donation",
        "email_donation",
        "cleaned_file",
        "first_name_contact",
        "last_name_contact",
        "email_contact",
    ]
]

review_table_subset.head()

,first_name_donation,last_name_donation,email_donation,cleaned_file,first_name_contact,last_name_contact,email_contact
0,Thomas,Babcock,NaN,cleaned_donation_source04_donation_great_futur...,Thomas,Babcock,NaN
1,Patrick,Carbone,NaN,cleaned_donation_source04_donation_great_futur...,Patrick,Carbone,NaN
2,Serge,Reiss,NaN,cleaned_donation_source04_donation_great_futur...,Serge,Reiss,NaN
3,Jacob & Rose Olum,Foundation,NaN,cleaned_donation_source04_donation_great_futur...,Jacob & Rose,Olum Foundation,NaN
4,Sharon,Corrington,NaN,cleaned_donation_source04_donation_great_futur...,Sharon,Corrington,NaN


**Review List Created:**

- Contains donation records matched to contacts by name-only.
- Displays donation-side and contact-side identity fields for comparison.
- These matches are NOT automatically confirmed.

Next Step:
Share this review list with mentor/stakeholders for validation before consolidation.